# 02_Modeling.ipynb - Model Training & Evaluation
## CS 451/551 Final Project: IVX Health No-Show Prediction

**Requirements Met:**
- ✅ 2+ model families (Logistic Regression + Random Forest + XGBoost)
- ✅ Train/Validation/Test split (70/15/15) with stratification
- ✅ Hyperparameter tuning via GridSearchCV
- ✅ Multiple metrics: Precision, Recall, F1, ROC-AUC
- ✅ Model comparison table + visualizations

In [10]:
# =============================================================================
# SECTION 1: Setup & Imports
# =============================================================================
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path for imports
sys.path.append('../src')
from data_loader import load_raw_data
from cleaning import clean_appointment_data

# Sklearn imports
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, f1_score, precision_score, 
                             recall_score, roc_curve)
from sklearn.preprocessing import StandardScaler

# Plot settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

## SECTION 2: Load & Prepare Data

In [11]:
# Load and clean data using our modular pipeline
print("Loading raw data...")
df = load_raw_data()

print("Applying cleaning pipeline...")
df_clean, cleaning_log = clean_appointment_data(df)

print(f"Cleaned data shape: {df_clean.shape}")
print(f"Target distribution: {df_clean['No-show'].value_counts(normalize=True).to_dict()}")

Loading raw data...
Applying cleaning pipeline...
Cleaned data shape: (71954, 19)
Target distribution: {'No': 0.7148316980292965, 'Yes': 0.28516830197070353}


In [12]:
# Select features for modeling
feature_cols = [
    'Age', 
    'lead_time_days', 
    'appointment_day_of_week', 
    'is_weekend', 
    'SMS_received', 
    'has_prior_conditions'
]

X = df_clean[feature_cols].copy()
y = df_clean['No-show'].copy()

print(f"Features: {feature_cols}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

Features: ['Age', 'lead_time_days', 'appointment_day_of_week', 'is_weekend', 'SMS_received', 'has_prior_conditions']
X shape: (71954, 6), y shape: (71954,)


## SECTION 3: Train/Validation/Test Split (No Data Leakage!)

In [13]:
# First split: 85% train+val, 15% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)

# Second split: 70% train, 15% val (from the 85%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val)

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

# Verify class distribution is preserved
print(f"\nTarget distribution preserved:")
print(f"Train: {y_train.mean():.3f} no-show rate")
print(f"Val: {y_val.mean():.3f} no-show rate")
print(f"Test: {y_test.mean():.3f} no-show rate")

Train: 50395 (70.0%)
Validation: 10765 (15.0%)
Test: 10794 (15.0%)

Target distribution preserved:


TypeError: Cannot perform reduction 'mean' with string dtype

In [ ]:
# Scale features for Logistic Regression (tree models don't need scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Keep unscaled for tree-based models
X_train_tree = X_train.copy()
X_val_tree = X_val.copy()
X_test_tree = X_test.copy()

## SECTION 4: Model 1 - Logistic Regression (Linear Family)

In [ ]:
# Baseline Logistic Regression
lr_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_base.fit(X_train_scaled, y_train)

# Hyperparameter tuning with GridSearchCV
lr_params = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['lbfgs', 'liblinear']
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    lr_params, cv=5, scoring='f1', n_jobs=-1)
lr_grid.fit(X_train_scaled, y_train)

print(f"Best LR params: {lr_grid.best_params_}")
print(f"Best LR CV F1: {lr_grid.best_score_:.3f}")

# Evaluate on test set
lr_best = lr_grid.best_estimator_
lr_pred = lr_best.predict(X_test_scaled)
lr_proba = lr_best.predict_proba(X_test_scaled)[:, 1]

## SECTION 5: Model 2 - Random Forest (Tree-Based Family)

In [ ]:
# Baseline Random Forest
rf_base = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_base.fit(X_train_tree, y_train)

# Hyperparameter tuning with GridSearchCV
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    rf_params, cv=5, scoring='f1', n_jobs=-1)
rf_grid.fit(X_train_tree, y_train)

print(f"Best RF params: {rf_grid.best_params_}")
print(f"Best RF CV F1: {rf_grid.best_score_:.3f}")

# Evaluate on test set
rf_best = rf_grid.best_estimator_
rf_pred = rf_best.predict(X_test_tree)
rf_proba = rf_best.predict_proba(X_test_tree)[:, 1]

## SECTION 6: Model 3 - Gradient Boosting (Optional Third Family)

In [ ]:
# Gradient Boosting (XGBoost-style)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train_tree, y_train)

gb_pred = gb.predict(X_test_tree)
gb_proba = gb.predict_proba(X_test_tree)[:, 1]

print("Gradient Boosting trained successfully")

## SECTION 7: Evaluate & Compare Models

In [ ]:
# Compile results for all models
results = []

models = [
    ('Logistic Regression', lr_pred, lr_proba),
    ('Random Forest', rf_pred, rf_proba),
    ('Gradient Boosting', gb_pred, gb_proba)
]

for name, pred, proba in models:
    results.append({
        'Model': name,
        'Accuracy': (pred == y_test).mean(),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1-Score': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, proba)
    })

results_df = pd.DataFrame(results)
print("\n=== Model Comparison (Test Set) ===")
print(results_df.to_string(index=False))

In [ ]:
# Identify best model by F1-Score (most important for imbalanced no-show prediction)
best_idx = results_df['F1-Score'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
print(f"\n🏆 Best Model: {best_model_name} (F1 = {results_df.loc[best_idx, 'F1-Score']:.3f})")

# Save best model name for deployment later
best_model_info = {
    'name': best_model_name,
    'f1': results_df.loc[best_idx, 'F1-Score'],
    'recall': results_df.loc[best_idx, 'Recall']
}
print(f"Best model info: {best_model_info}")

## SECTION 8: Visualizations

In [ ]:
# Figure 1: Confusion Matrix - Best Model
plt.figure(figsize=(8, 6))
if best_model_name == 'Logistic Regression':
    cm = confusion_matrix(y_test, lr_pred)
else:
    cm = confusion_matrix(y_test, rf_pred)  # Use RF as default for viz

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Show', 'No-Show'],
            yticklabels=['Show', 'No-Show'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('../report/figures/fig_confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
# Figure 2: ROC Curves - All Models
plt.figure(figsize=(10, 8))

for name, pred, proba in models:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../report/figures/fig_roc_curves.png', dpi=300)
plt.show()

In [ ]:
# Figure 3: Feature Importance - Random Forest
if best_model_name in ['Random Forest', 'Gradient Boosting']:
    importances = rf_best.feature_importances_ if best_model_name == 'Random Forest' else gb.feature_importances_
    feat_imp = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': importances
    }).sort_values('Importance', ascending=True)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feat_imp['Feature'], feat_imp['Importance'])
    plt.xlabel('Importance')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.tight_layout()
    plt.savefig('../report/figures/fig_feature_importance.png', dpi=300)
    plt.show()
    
    print("\nTop 3 Most Important Features:")
    print(feat_imp.tail(3).to_string(index=False))

## SECTION 9: Classification Report - Best Model

In [ ]:
# Detailed classification report for best model
print(f"\n=== Classification Report: {best_model_name} ===")
if best_model_name == 'Logistic Regression':
    print(classification_report(y_test, lr_pred, target_names=['Show', 'No-Show']))
elif best_model_name == 'Random Forest':
    print(classification_report(y_test, rf_pred, target_names=['Show', 'No-Show']))
else:
    print(classification_report(y_test, gb_pred, target_names=['Show', 'No-Show']))

## SECTION 10: Save Best Model for Deployment

In [ ]:
import joblib

# Save the best model + scaler + feature list for deployment
model_artifacts = {
    'model': lr_best if best_model_name == 'Logistic Regression' else (rf_best if best_model_name == 'Random Forest' else gb),
    'scaler': scaler,
    'feature_cols': feature_cols,
    'best_model_name': best_model_name,
    'metrics': best_model_info
}

output_path = Path('../models/best_model.pkl')
output_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model_artifacts, output_path)
print(f"✅ Best model saved to {output_path}")
print(f"   Model: {best_model_info}")

## SECTION 11: Summary for Report

In [ ]:
# Print summary for copying into IEEE report
print("\n" + "="*60)
print("REPORT SUMMARY - Copy into Section 6: Results")
print("="*60)
print(f"\nWe evaluated three model families on a held-out test set (15% of data):")
print(f"\n{results_df.to_markdown(index=False)}")
print(f"\nBest performing model: **{best_model_name}**")
print(f"- F1-Score: {best_model_info['f1']:.3f}")
print(f"- Recall: {best_model_info['recall']:.3f} (critical for catching no-shows)")
print(f"- ROC-AUC: {results_df[results_df['Model']==best_model_name]['ROC-AUC'].values[0]:.3f}")
print(f"\nKey insight: {feature_cols[np.argmax(rf_best.feature_importances_)]} was the most predictive feature.")
print("\n" + "="*60)